In [1]:
import numpy as np
np.set_printoptions(suppress = True)

In [2]:
# get randomly distributed parameters
params = np.random.uniform(
    low = -50 , high = 150 , size = 20
)
params

array([-32.70367891, 149.74879834,  53.1197357 ,  72.08484289,
       -48.94881329, -44.84824491, -39.64063685,  -7.19825507,
         1.74379448,  96.09979771,  20.37061153,  18.02162261,
         2.84764255,  -4.62146558, -27.32639003,  98.54758343,
        -0.84440553, 115.34095395, 144.54065983,  -9.82875057])

In [3]:
params[0] = params.max() + 1
params[1] = params.min() - 1
params[2] = 0

In [4]:
# round each numbers to the second decimal place
params = np.round(params , 2)
params

array([150.75, -49.95,   0.  ,  72.08, -48.95, -44.85, -39.64,  -7.2 ,
         1.74,  96.1 ,  20.37,  18.02,   2.85,  -4.62, -27.33,  98.55,
        -0.84, 115.34, 144.54,  -9.83])

### Define the quantization

In [5]:
def clamp(params_q: np.array , lower_bound: int , upper_bound: int) -> np.array:
    params_q[params_q < lower_bound] = lower_bound
    params_q[params_q > upper_bound] = upper_bound
    return params_q

In [6]:
def asymmetric_quantization(params: np.array , bits: int) -> tuple[np.array , float , int]:
    # calculate the scale and zero 
    alpha = np.max(params)
    beta = np.min(params)
    scale = (alpha - beta) / (2**bits - 1)
    zero = -1 * np.round(beta / scale)

    lower_bound = 0
    upper_bound = 2**bits - 1
    quantized = clamp(
        np.round(params / scale + zero), lower_bound , upper_bound
    ).astype(np.int32)
    return quantized , scale , zero

In [7]:
def symmetric_quantization(params: np.array ,bits: int)-> tuple[np.array , float]: 
    alpha = np.max(np.abs(params))
    scale = alpha / 2**(bits - 1)
    lower_bound = -2**(bits - 1)
    upper_bound = 2**(bits - 1) - 1
    quantized = clamp(np.round(params/scale) , lower_bound , upper_bound).astype(np.int32)
    return quantized , scale 

In [8]:
def quantization_error(params: np.array , params_q: np.array): 
    return np.mean((params - params_q) ** 2)

In [9]:
(asymmetric_q , asymmetric_scale , asymmetric_zero) = asymmetric_quantization(
    params , bits = 8
)
(symmetric_q , symmetric_scale) = symmetric_quantization(
    params , 8
)

In [10]:
print(f"Original: {np.round(params , 2)}")
print(' - ' * 30)
print(f"Asymmetric scale: {asymmetric_scale} , Asymmetric zero: {asymmetric_zero}")
print(f"Asymmetric quantized: {asymmetric_q}")
print(' - ' * 30)

print(f"symmetric scale: {symmetric_scale}")
print(f"symmetric quantized: {symmetric_q}")

Original: [150.75 -49.95   0.    72.08 -48.95 -44.85 -39.64  -7.2    1.74  96.1
  20.37  18.02   2.85  -4.62 -27.33  98.55  -0.84 115.34 144.54  -9.83]
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
Asymmetric scale: 0.7870588235294117 , Asymmetric zero: 63.0
Asymmetric quantized: [255   0  63 155   1   6  13  54  65 185  89  86  67  57  28 188  62 210
 247  51]
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
symmetric scale: 1.177734375
symmetric quantized: [127 -42   0  61 -42 -38 -34  -6   1  82  17  15   2  -4 -23  84  -1  98
 123  -8]
